# EE 451: Communications Systems
## Lesson 11 - FM/PM Theory & Binary FSK

### Learning Objectives
By the end of this lesson, you will be able to:
- Apply Bessel functions to calculate FM spectrum components
- Use Carson's rule to estimate FM bandwidth
- Explain Binary FSK as the digital counterpart to FM
- Analyze Continuous Phase FSK (CPFSK) and MSK
- Compare FM and FSK spectral characteristics

### Textbook Reference
Haykin & Moher, Chapter 4.1-4.5

In [ ]:
# Setup: Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
from scipy.special import jv  # Bessel function of the first kind
import warnings
warnings.filterwarnings('ignore')

# Use consistent plot style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2

print("Setup complete! NumPy version:", np.__version__)

## Part 1: Bessel Functions

For single-tone FM: $s(t) = A_c \cos[2\pi f_c t + \beta \sin(2\pi f_m t)]$

**Bessel expansion:** Spectrum contains discrete components:
- Carrier at $f_c$: amplitude $A_c J_0(\beta)$
- Sidebands at $f_c \pm n f_m$: amplitude $A_c J_n(\beta)$

**Key properties of $J_n(\beta)$:**
- $J_0(0) = 1$, $J_n(0) = 0$ for $n > 0$ (unmodulated carrier)
- Small $\beta$: $J_0 \approx 1$, $J_1 \approx \beta/2$ (NBFM)
- Carrier null at $\beta = 2.405, 5.520, 8.654, \ldots$

In [ ]:
# Part 1: Bessel Functions of the First Kind

beta_range = np.linspace(0, 12, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: J_n(beta) vs beta ---
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 0.6, 6))
for n in range(6):
    ax.plot(beta_range, jv(n, beta_range), color=colors[n], linewidth=2,
            label=f'J_{n}(\u03b2)')

# Mark carrier nulls (J_0 = 0)
nulls = [2.405, 5.520, 8.654, 11.792]
for z in nulls:
    if z < 12:
        ax.plot(z, 0, 'kx', markersize=10, markeredgewidth=2)

ax.axhline(y=0, color='gray', linewidth=0.5)
ax.set_xlabel('Modulation Index \u03b2')
ax.set_ylabel('J_n(\u03b2)')
ax.set_title('Bessel Functions of the First Kind')
ax.legend(fontsize=10, ncol=2)

# --- Plot 2: Power in components for specific beta ---
ax = axes[1]
beta_ex = 3.0
n_vals = np.arange(0, 10)
Jn_vals = jv(n_vals, beta_ex)

# Total power check: J_0^2 + 2*sum(J_n^2) should = 1
power_check = Jn_vals[0]**2 + 2 * np.sum(Jn_vals[1:]**2)

bars = ax.bar(n_vals, np.abs(Jn_vals), color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Sideband Order n')
ax.set_ylabel('|J_n(\u03b2)|')
ax.set_title(f'Bessel Coefficients for \u03b2 = {beta_ex}')
ax.set_xticks(n_vals)

for bar, val in zip(bars, Jn_vals):
    if abs(val) > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, abs(val) + 0.01,
                f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# Print Bessel function table
print("=" * 55)
print("Bessel Function Table: J_n(beta)")
print("=" * 55)
print(f"{'beta':>5}", end='')
for n in range(7):
    print(f"  {'J_'+str(n):>7}", end='')
print()
print("-" * 61)
for beta in [0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 8.0]:
    print(f"{beta:>5.2f}", end='')
    for n in range(7):
        print(f"  {jv(n, beta):>7.3f}", end='')
    print()
print(f"\nCarrier nulls (J_0 = 0): beta = {', '.join(f'{z:.3f}' for z in nulls)}")

## Part 2: FM Spectrum

Single-tone FM spectrum has discrete lines:
- Line at $f_c + n f_m$ with amplitude $\propto J_n(\beta)$
- Infinite sidebands in theory, but amplitude drops for large $n$

**Practical bandwidth:** Include sidebands where $|J_n(\beta)| > 0.01$ (1% of unmodulated carrier)

In [ ]:
# Part 2: FM Spectrum at Different Modulation Indices

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
beta_values = [0.5, 1.0, 2.0, 5.0]
fm = 1000  # 1 kHz message for labeling

for idx, (beta, ax) in enumerate(zip(beta_values, axes.flatten())):
    n_max = int(beta + 5)  # Number of sidebands to show
    n_vals = np.arange(-n_max, n_max + 1)
    
    # Bessel coefficients (J_{-n} = (-1)^n J_n)
    amplitudes = [jv(abs(n), beta) for n in n_vals]
    
    # Stem plot
    markerline, stemlines, baseline = ax.stem(n_vals, amplitudes, basefmt='gray')
    markerline.set_markersize(6)
    stemlines.set_linewidth(2)
    
    # Color code: positive = blue, negative = red
    for n, amp in zip(n_vals, amplitudes):
        color = 'steelblue' if amp >= 0 else 'coral'
        ax.plot(n, amp, 'o', color=color, markersize=8)
    
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.set_xlabel('Sideband Order n  (freq = f_c + n\u00b7f_m)')
    ax.set_ylabel('J_n(\u03b2)')
    
    delta_f = beta * fm
    bw = 2 * (delta_f + fm)
    ax.set_title(f'\u03b2 = {beta}, \u0394f = {delta_f/1000:.1f} kHz, '
                 f'Carson BW = {bw/1000:.0f} kHz')

plt.tight_layout()
plt.show()

print("=" * 55)
print("FM Spectrum: Number of Significant Sidebands")
print("=" * 55)
for beta in beta_values:
    # Count sidebands with |J_n| > 0.01
    n_sig = 0
    for n in range(1, 20):
        if abs(jv(n, beta)) > 0.01:
            n_sig = n
    print(f"beta = {beta}: {n_sig} significant sideband pairs")

## Part 3: Carson's Rule

**Carson's Rule:** $BW \approx 2(\Delta f + f_m) = 2(\beta + 1) f_m$

- Captures ~98% of signal power
- For large $\beta$: $BW \approx 2\Delta f$ (dominated by frequency swing)
- For small $\beta$: $BW \approx 2 f_m$ (like AM)

**FM Broadcast:** $\Delta f = 75$ kHz, $f_m = 15$ kHz $\Rightarrow BW = 180$ kHz (FCC: 200 kHz channels)

In [ ]:
# Part 3: Carson's Rule - Bandwidth vs Modulation Index

fm = 15e3  # 15 kHz (FM broadcast audio bandwidth)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Carson's rule BW vs beta ---
ax = axes[0]
beta_range = np.linspace(0.1, 10, 200)
bw_carson = 2 * (beta_range + 1) * fm / 1e3  # in kHz
bw_2fm = 2 * fm / 1e3 * np.ones_like(beta_range)  # AM bandwidth
bw_2deltaf = 2 * beta_range * fm / 1e3  # Large-beta approx

ax.plot(beta_range, bw_carson, 'b-', linewidth=2, label="Carson: 2(\u0394f + f_m)")
ax.plot(beta_range, bw_2fm, 'g--', linewidth=1.5, label='AM BW: 2f_m')
ax.plot(beta_range, bw_2deltaf, 'r:', linewidth=1.5, label='Large \u03b2: 2\u0394f')

# Mark FM broadcast point
beta_fm = 75 / 15  # = 5
bw_fm = 2 * (75 + 15)  # = 180 kHz
ax.plot(beta_fm, bw_fm, 'ro', markersize=10)
ax.annotate(f'FM Broadcast\n\u03b2={beta_fm}, BW={bw_fm} kHz',
            xy=(beta_fm, bw_fm), xytext=(beta_fm + 1.5, bw_fm - 30),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='red'),
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('Modulation Index \u03b2')
ax.set_ylabel('Bandwidth (kHz)')
ax.set_title("Carson's Rule: BW vs \u03b2 (f_m = 15 kHz)")
ax.legend()

# --- Plot 2: Power contained within Carson's BW ---
ax = axes[1]
betas = np.arange(0.5, 8.5, 0.5)
power_fraction = []

for beta in betas:
    n_carson = int(np.ceil(beta + 1))  # Number of sideband pairs in Carson BW
    total_power = jv(0, beta)**2 + 2 * sum(jv(n, beta)**2 for n in range(1, 30))
    carson_power = jv(0, beta)**2 + 2 * sum(jv(n, beta)**2 for n in range(1, n_carson + 1))
    power_fraction.append(carson_power / total_power)

ax.bar(betas, power_fraction, width=0.4, color='steelblue', alpha=0.7, edgecolor='black')
ax.axhline(y=0.98, color='r', linestyle='--', label='98% power')
ax.set_xlabel('Modulation Index \u03b2')
ax.set_ylabel('Fraction of Power in Carson BW')
ax.set_title("Power Captured by Carson's Rule")
ax.set_ylim(0.9, 1.01)
ax.legend()

plt.tight_layout()
plt.show()

# Print FM broadcast example
print("=" * 55)
print("FM Broadcast Example")
print("=" * 55)
print(f"Max frequency deviation: Delta_f = 75 kHz")
print(f"Max audio frequency: fm = 15 kHz")
print(f"Modulation index: beta = 75/15 = {75/15:.0f}")
print(f"Carson's rule: BW = 2(75 + 15) = 180 kHz")
print(f"FCC channel spacing: 200 kHz (includes guard bands)")
print(f"Compare to AM: BW = 2 * 5 = 10 kHz (FM uses 18x more spectrum!)")

## Part 4: Binary Frequency Shift Keying (FSK)

**FSK:** Digital counterpart to analog FM
- Bit "1" (mark): $f_1 = f_c + \Delta f$
- Bit "0" (space): $f_0 = f_c - \Delta f$
- Frequency separation: $2\Delta f$

**Modulation index:** $h = 2\Delta f \cdot T_b = 2\Delta f / R_b$

**CPFSK:** Phase continuity at bit transitions reduces spectral spreading

**MSK:** Special CPFSK with $h = 0.5$ (minimum orthogonal spacing)

In [ ]:
# Part 4: Binary FSK Signal Generation

# Parameters
Rb = 1000     # Bit rate (bps)
Tb = 1 / Rb   # Bit period (s)
fc = 5000     # Carrier frequency (Hz)
delta_f = 500 # Frequency deviation (Hz)
fs = 100000   # Sampling rate

f1 = fc + delta_f  # Frequency for bit '1'
f0 = fc - delta_f  # Frequency for bit '0'
h = 2 * delta_f / Rb  # Modulation index

# Data sequence
bits = np.array([1, 0, 1, 1, 0, 0, 1, 0])
N_bits = len(bits)

# Generate CPFSK signal (phase continuous)
samples_per_bit = int(Tb * fs)
t_total = np.arange(N_bits * samples_per_bit) / fs

# Frequency signal
freq_signal = np.zeros(len(t_total))
for i, bit in enumerate(bits):
    start = i * samples_per_bit
    end = (i + 1) * samples_per_bit
    freq_signal[start:end] = f1 if bit == 1 else f0

# CPFSK: Integrate frequency to get phase
phase = 2 * np.pi * np.cumsum(freq_signal) / fs
s_fsk = np.cos(phase)

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

# Binary data
bit_signal = np.repeat(bits, samples_per_bit)
axes[0].step(t_total * 1000, bit_signal, 'b-', linewidth=2, where='post')
axes[0].set_ylabel('Bit Value')
axes[0].set_title(f'Binary Data: {list(bits)}')
axes[0].set_yticks([0, 1])
axes[0].set_ylim(-0.2, 1.3)

# Instantaneous frequency
axes[1].step(t_total * 1000, freq_signal, 'r-', linewidth=2, where='post')
axes[1].axhline(y=fc, color='gray', linestyle='--', alpha=0.5, label=f'f_c = {fc} Hz')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_title(f'Instantaneous Frequency (f_0={f0} Hz, f_1={f1} Hz)')
axes[1].legend()

# FSK waveform
axes[2].plot(t_total * 1000, s_fsk, 'purple', linewidth=0.8)
# Add bit boundaries
for i in range(1, N_bits):
    axes[2].axvline(x=i * Tb * 1000, color='gray', linestyle=':', alpha=0.3)
axes[2].set_ylabel('s(t)')
axes[2].set_xlabel('Time (ms)')
axes[2].set_title(f'CPFSK Waveform (h = {h:.1f})')

plt.tight_layout()
plt.show()

print("=" * 55)
print("FSK Parameters")
print("=" * 55)
print(f"Bit rate: Rb = {Rb} bps")
print(f"Carrier: fc = {fc} Hz")
print(f"Deviation: Delta_f = {delta_f} Hz")
print(f"f_0 (bit 0) = {f0} Hz, f_1 (bit 1) = {f1} Hz")
print(f"Modulation index: h = 2*Delta_f/Rb = {h:.1f}")
print(f"FSK bandwidth (Carson): BW = 2(Delta_f + Rb) = {2*(delta_f + Rb)} Hz")

## Part 5: FM vs FSK Comparison

| Property | Analog FM | Binary FSK |
|----------|-----------|------------|
| Message | Continuous $m(t)$ | Binary bits |
| Frequency | $f_c + k_f m(t)$ | $f_0$ or $f_1$ |
| Mod. index | $\beta = \Delta f / f_m$ | $h = 2\Delta f / R_b$ |
| Bandwidth | $2(\Delta f + f_m)$ | $2(\Delta f + R_b)$ |
| Envelope | Constant | Constant |

**Applications:** FM broadcast, FSK in modems/IoT, MSK in GSM/satellite

In [ ]:
# Part 5: FSK Spectrum and FM vs FSK

# Generate longer FSK signal for spectrum analysis
np.random.seed(42)
N_bits_spec = 1000
bits_long = np.random.randint(0, 2, N_bits_spec)

fs = 100000
Rb = 1000
fc = 10000
samples_per_bit = int(fs / Rb)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: FSK spectra at different modulation indices ---
ax = axes[0]
h_values = [0.5, 1.0, 2.0]
colors = ['steelblue', 'coral', 'green']

for h, color in zip(h_values, colors):
    delta_f = h * Rb / 2
    
    # Generate CPFSK
    freq_sig = np.zeros(N_bits_spec * samples_per_bit)
    for i, bit in enumerate(bits_long):
        start = i * samples_per_bit
        end = (i + 1) * samples_per_bit
        freq_sig[start:end] = fc + delta_f if bit else fc - delta_f
    
    phase = 2 * np.pi * np.cumsum(freq_sig) / fs
    s = np.cos(phase)
    
    # Spectrum
    N_fft = len(s)
    S = fft(s * np.hanning(N_fft)) / N_fft
    freqs = fftfreq(N_fft, 1/fs)
    
    mask = (freqs >= fc - 4000) & (freqs <= fc + 4000)
    psd = 20 * np.log10(np.abs(S[mask]) + 1e-10)
    psd -= np.max(psd)  # Normalize
    
    ax.plot((freqs[mask] - fc) / 1000, psd, color=color, linewidth=1.5,
            label=f'h = {h} (MSK)' if h == 0.5 else f'h = {h}')

ax.set_xlabel('Frequency offset from f_c (kHz)')
ax.set_ylabel('PSD (dB, normalized)')
ax.set_title('CPFSK Spectrum: Different Modulation Indices')
ax.set_ylim(-40, 5)
ax.legend()

# --- Plot 2: Bandwidth comparison ---
ax = axes[1]
h_range = np.linspace(0.25, 5, 100)
bw_fsk = 2 * (h_range * Rb / 2 + Rb) / 1000  # Carson-like for FSK, in kHz
bw_ask = 2 * Rb / 1000 * np.ones_like(h_range)  # ASK bandwidth

ax.plot(h_range, bw_fsk, 'b-', linewidth=2, label='FSK: 2(\u0394f + R_b)')
ax.plot(h_range, bw_ask, 'g--', linewidth=2, label='ASK: 2R_b')
ax.axvline(x=0.5, color='r', linestyle=':', label='MSK (h=0.5)')

ax.set_xlabel('FSK Modulation Index h')
ax.set_ylabel('Bandwidth (kHz)')
ax.set_title(f'FSK Bandwidth vs Modulation Index (R_b = {Rb} bps)')
ax.legend()

plt.tight_layout()
plt.show()

print("=" * 55)
print("FSK Bandwidth Comparison")
print("=" * 55)
print(f"Bit rate: Rb = {Rb} bps")
for h in [0.5, 1.0, 2.0]:
    df = h * Rb / 2
    bw = 2 * (df + Rb)
    print(f"\n  h = {h}: Delta_f = {df:.0f} Hz, BW = {bw:.0f} Hz")
    if h == 0.5:
        print(f"    (MSK: minimum spacing for orthogonality)")

## Summary

### Key Formulas

| Concept | Formula |
|---------|--------|
| FM spectrum | Carrier: $A_c J_0(\beta)$, Sidebands: $A_c J_n(\beta)$ at $f_c \pm nf_m$ |
| Carson's rule | $BW \approx 2(\Delta f + f_m) = 2(\beta + 1)f_m$ |
| FSK frequencies | $f_1 = f_c + \Delta f$, $f_0 = f_c - \Delta f$ |
| FSK mod. index | $h = 2\Delta f / R_b$ |
| MSK condition | $h = 0.5$, $\Delta f = R_b/4$ |

### Key Takeaways
1. **Bessel functions** determine FM spectral components — carrier and sideband amplitudes depend on $\beta$
2. **Carson's rule** captures ~98% of FM power: $BW \approx 2(\Delta f + f_m)$
3. **FSK** is the digital version of FM — two discrete frequencies for binary data
4. **CPFSK** maintains phase continuity, giving a more compact spectrum
5. **MSK** ($h = 0.5$) provides minimum orthogonal tone spacing — used in GSM

### Next Topics
- **Lesson 12:** FSK demodulation (coherent, non-coherent, PLL)